> **Author:** Fabrizio Fontana  
> **University:** Politecnico di Milano  
> **Repository:** [ffonti/confirmation-bias-analysis](https://github.com/ffonti/confirmation-bias-analysis)  
> **Supervisor:** Prof. Cinzia Cappiello  
> **Co-supervisor:** Dott. Mattia Sabella

# **NLI Evaluation**
This notebook measures the *Confirmation Bias* using the `deberta-v3-large` model as a Cross-Encoder to derive Entailment metrics.

In [ ]:
import sys
import os
import pandas as pd

# Add the parent directory to the system path to allow imports from the src directory
sys.path.append(os.path.abspath('..'))

from src.evaluators.nli import compute_nli_metrics

## **Data Loading**
Upload one of the logs generated in the pipeline located in the `data/interim/` folder.

In [ ]:
# Define the path to the generated results (JSONL format)
file_path = "../data/interim/3_fever_deepseek_r1_results.jsonl"
# file_path = "../data/interim/4_truthfulqa_deepseek_r1_results.jsonl"
# file_path = "../data/interim/5_mmlu_pro_deepseek_r1_results.jsonl"

try:
    # Load the generated results into a DataFrame
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples.")
    display(df_results.head(1))
except FileNotFoundError:
    print(f"Il file non è stato trovato al path richiesto ({file_path}).")

## **Evaluation**
Compute the evaluation metrics by sending the generated results to the model, that processes the Entailment Score.

In [ ]:
# Compute the NLI metrics
df_evaluated = compute_nli_metrics(df_results)

## **Summary**
Compute summary statistics on the evaluated results, and save the final DataFrame to `data/processed/` for later use.

In [ ]:
# Aggregation of the average scores for each metric
mean_combined = df_evaluated["cb_combined"].mean()
mean_shift = df_evaluated["cb_shift"].mean()
mean_self = df_evaluated["cb_self"].mean()

print("Results of the NLI Evaluation:")
print(f"Confirmation Bias Mean (Combined):  {mean_combined:.6f}")
print(f"Average Stance Shift Drop:          {mean_shift:.6f}")
print(f"Average Self-Contradiction Drop:    {mean_self:.6f}")

display(df_evaluated[["sample", "cb_shift", "cb_self", "cb_combined"]].head(10))

## **Exporting**
Export the results to a CSV file for later use in the final analysis notebook.

In [ ]:
# Extract filename from the input path
base_name = os.path.basename(file_path).replace("_results.jsonl", "")
output_file = f"../data/interim/{base_name}_nli.csv"

# Save the evaluated DataFrame to the interim data directory
df_evaluated.to_csv(output_file, index=False)
print(f"Saved evaluation file to {output_file}")